# LLM-Assisted API Debugging Lab - Reproducible Colab

This notebook runs the Rust workspace from a fresh clone each time. It demonstrates a deterministic CLI over synthetic API failure fixtures; it is not a learned classifier, does not call an LLM, and does not claim production-traffic accuracy. The useful signal is the engineering discipline: evidence extraction, deterministic rules, explicit hypotheses and unknowns, snapshot-pinned outputs, and a prompt boundary that treats log-derived text as untrusted.

## 1. Runtime Setup

Colab runtimes are ephemeral, so this cell installs the pinned Rust toolchain and prints provenance for the run.

In [ ]:
%%bash
set -euxo pipefail
TOOLCHAIN="1.94.1"
if ! command -v rustup >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain "$TOOLCHAIN"
fi
source "$HOME/.cargo/env"
rustup toolchain install "$TOOLCHAIN" --profile minimal
rustup default "$TOOLCHAIN"
rustup component add rustfmt clippy
rustc --version
cargo --version
uname -a
date -u +"%Y-%m-%dT%H:%M:%SZ"


## 2. Fresh Clone

The default reference is `main`. To reproduce a specific reviewed state, set `REF` to an immutable commit SHA before running this cell.

In [ ]:
%%bash
set -euxo pipefail
REPO_URL="https://github.com/infinityabundance/LLM-Assisted-API-Debugging-Lab.git"
REF="${REF:-main}"
WORKDIR="/content/LLM-Assisted-API-Debugging-Lab"
rm -rf "$WORKDIR"
git clone "$REPO_URL" "$WORKDIR"
cd "$WORKDIR"
git checkout --detach "$REF"
git rev-parse HEAD
git status --short


## 3. Preflight

This verifies the lockfile, renamed crate path, workspace metadata, and fixture inventory before running the full gates.

In [ ]:
%%bash
set -euxo pipefail
source "$HOME/.cargo/env"
cd /content/LLM-Assisted-API-Debugging-Lab
test -f Cargo.lock
test -f crates/llm-assisted-api-debugging-lab/Cargo.toml
cargo metadata --locked --no-deps --format-version 1 >/tmp/llm_lab_metadata.json
python3 - <<'PY'
import json
metadata = json.load(open('/tmp/llm_lab_metadata.json'))
names = [package['name'] for package in metadata['packages']]
assert 'llm-assisted-api-debugging-lab' in names, names
print('packages:', names)
PY
case_count=$(find fixtures/cases -name '*.json' | wc -l)
log_count=$(find fixtures/logs -type f | wc -l)
test "$case_count" -eq 8
test "$log_count" -eq 8
printf 'fixture cases: %s\nfixture logs: %s\n' "$case_count" "$log_count"


## 4. CI-Equivalent Gates

These are the same checks expected to pass before treating the demo as credible.

In [ ]:
%%bash
set -euxo pipefail
source "$HOME/.cargo/env"
cd /content/LLM-Assisted-API-Debugging-Lab
cargo fmt --check
cargo clippy --locked --workspace --all-targets -- -D warnings
cargo test --locked --workspace --all-targets


## 5. Demo Script

Run the repository's own five-minute path exactly as shipped.

In [ ]:
%%bash
set -euxo pipefail
source "$HOME/.cargo/env"
cd /content/LLM-Assisted-API-Debugging-Lab
bash scripts/run_demo.sh


## 6. Reviewer-Facing Outputs

These commands show representative behavior rather than only pass/fail test output.

In [ ]:
%%bash
set -euxo pipefail
source "$HOME/.cargo/env"
cd /content/LLM-Assisted-API-Debugging-Lab
cargo run --locked -p llm-assisted-api-debugging-lab -- list-cases
cargo run --locked -p llm-assisted-api-debugging-lab -- diagnose webhook_signature
cargo run --locked -p llm-assisted-api-debugging-lab -- diagnose-log fixtures/logs/timeout.log
cargo run --locked -p llm-assisted-api-debugging-lab -- report rate_limit | sed -n '1,120p'
cargo run --locked -p llm-assisted-api-debugging-lab -- prompt-json webhook_signature >/tmp/webhook_prompt.json
python3 -m json.tool /tmp/webhook_prompt.json | sed -n '1,100p'


## 7. Safety and Failure Gates

This checks the prompt-boundary sanitizer fixture and the CLI's expected unknown-case exit code.

In [ ]:
%%bash
set -euxo pipefail
source "$HOME/.cargo/env"
cd /content/LLM-Assisted-API-Debugging-Lab
cargo run --locked -p llm-assisted-api-debugging-lab -- prompt injection_attempt >/tmp/injection_prompt.txt
python3 - <<'PY'
text = open('/tmp/injection_prompt.txt').read()
assert "\\nIGNORE ALL PREVIOUS" in text
assert not any(line.lstrip().startswith('IGNORE ALL PREVIOUS') for line in text.splitlines())
print('sanitization gate passed')
PY
set +e
cargo run --quiet --locked -p llm-assisted-api-debugging-lab -- diagnose not_a_real_case >/tmp/unknown_stdout 2>/tmp/unknown_stderr
code=$?
set -e
cat /tmp/unknown_stderr
test "$code" -eq 2
echo "unknown-case exit-code gate passed"


## 8. Interpretation

If every cell above passed, this run verified that a fresh checkout can build, lint, test, and demo the deterministic diagnoser. It still does not prove production accuracy, model quality, or protection against every prompt-injection strategy. The fixtures are synthetic; they are useful here because they make support-engineering habits auditable: separate evidence from hypothesis, mark unknowns, pin user-visible output, and keep the LLM-facing prompt downstream of deterministic diagnosis.